# Disclaimer!
The data Luke uses in this lecture are very big data files. We have made smaller versions available to you. There may be slight differences between Luke's lecture and this notebook, but the concepts are all the same.

# Lesson 2: Loading and Visualizing Spatial Data in R Pt. 1

This notebook introduces the fundamental concepts and techniques for working with spatial data in R, focusing on both **raster** and **vector** data types.

## Learning Objectives:
- Understand the basic structure and properties of raster and vector data
- Load raster and vector data in R
- Learn the basics of the `sf` and `terra` packages
- Explore metadata and attributes of spatial data
- Create basic visualizations of spatial data

Let's get started by loading the necessary packages!

In [ ]:
# run this line of code to quickly install the necessary spatial packages
system('apt-get install r-cran-sf r-cran-terra r-cran-tidyterra r-cran-knitr')



In [ ]:
# Load the required packages
library(sf)          # Simple Features - a standardized way to encode spatial vector data
library(ggplot2)     # Data visualization package
library(dplyr)       # Data manipulation package
library(terra)       # Spatial data manipulation and analysis
library(tidyterra)   # Integration of terra with the tidyverse
library(knitr)       # For nice table output

# set maximum memory usage limit for as long as your current kernel remains active
# this line has been added to your code-along notebook on top of Luke's version to adjust for computing specification differences
terraOptions(memmax = 3.5)

## Part 1: Understanding and Visualizing Raster Data

### What is Raster Data?

Raster data represents the world as a grid of cells. Each cell (or pixel) contains a value that represents information, such as:
- Elevation
- Temperature
- Land cover type
- Forest cover percentage
- Precipitation

Key components of raster data include:
- **Resolution**: Size of each cell (e.g., 30m x 30m)
- **Extent**: Geographic boundaries of the entire raster
- **CRS**: Coordinate Reference System that defines how coordinates map to locations on Earth
- **Values**: Data stored in each cell
- **Bands/Layers**: Multiple variables can be stored in separate bands (e.g., RGB channels in a satellite image)

### Loading a Raster File

Let's load the Hansen Global Forest Change dataset, which contains information about tree cover and forest loss:

In [ ]:
# Load the Hansen Global Forest Change dataset as a raster
hansen <- terra::rast("https://github.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/raw/refs/heads/main/course-4-environmental-analysis/geospatial-data/raster/hansen.tif")

# Let's examine the basic properties of this raster
print(hansen)


### The Terra Package for Raster Data

The `terra` package is a powerful R package for spatial data analysis, particularly for working with raster data. It's designed as a modern replacement for the older `raster` package, offering improved performance and functionality.

Key features of the `terra` package include:
- **Efficient handling** of large raster datasets with lower memory usage
- **Fast processing** of spatial operations
- **Support for multi-layer** raster data (SpatRaster objects)
- **Integration with other spatial packages** like `sf` for vector data
- **Comprehensive functions** for raster manipulation, analysis, and visualization

In this course, we'll use `terra` to:

1. Load and inspect raster data
2. Extract and manipulate raster values
3. Perform basic raster calculations
4. Visualize raster data using both base R plotting and `ggplot2` via `tidyterra`

The `tidyterra` package extends `terra` functionality to work seamlessly with the tidyverse ecosystem, making it easier to create beautiful visualizations with `ggplot2`.


### Exploring Raster Metadata

Understanding the metadata of your raster is crucial before analysis. Let's extract key information:

In [ ]:
# Examine raster properties
cat("Dimensions (rows, cols, layers):", dim(hansen), "\n")
cat("Resolution (x, y):", res(hansen), "\n")
cat("Extent:", as.character(ext(hansen)), "\n")
cat("Available layers:", names(hansen), "\n")
cat("Coordinate Reference System:", crs(hansen), "\n")


### Working with Raster Bands

Many raster datasets contain multiple bands (or layers). Each band typically represents a different variable. Let's extract individual bands from our Hansen dataset. We're mostly interested in the places that were forested in 2000 and places that have lost forest since (including which year they lost forest):

In [ ]:
# Extract specific bands
treecover2000 <- hansen$treecover2000  # Tree cover percentage in the year 2000
lossyear <- hansen$lossyear            # Year of forest loss (if any)

# Set both bands to NA where datamask is not equal to 1
# The datamask is 0 for oceans and 2 for non-ocean water bodies.
datamask <- hansen$datamask
treecover2000[datamask != 1] <- NA
lossyear[datamask != 1] <- NA
# While those places do have 0 forest and forest loss, let's remove them because they aren't really eligible to have forest.
# Examine the treecover2000 band
print(treecover2000)

### Basic Raster Visualization

The simplest way to visualize a raster is using the `plot()` function from the terra package:

In [ ]:
# Basic plot using terra
plot(treecover2000, main="Tree Cover in 2000", col=colorRampPalette(c("black", "green"))(100))

# try plotting the datamask band to verify that 0 is oceans and 2 is non-ocean water bodies
# YOUR CODE HERE

### Advanced Raster Visualization with ggplot2

For more customized and publication-quality visualizations, we can use ggplot2 with the tidyterra package:

In [ ]:
# Create a more aesthetically pleasing plot with ggplot2
ggplot() +
  geom_spatraster(data = treecover2000) +
  scale_fill_gradientn(
    colors = c("white", "darkgreen"),
    na.value = "transparent",
    name = "Tree Cover %"
  ) +
  theme_minimal() +
  labs(title = "Tree Cover in 2000",
       subtitle = "Percentage of tree canopy coverage per pixel",
       caption = "Source: Hansen Global Forest Change Data")

## Exercise 1: Exploring Raster Data

Now it's your turn to practice working with raster data!

1. Create a new visualization of the `lossyear` band from the Hansen dataset.
2. Use a color palette that makes it easy to distinguish different years of forest loss.
3. Add appropriate title and labels to your plot.

**Hint**: The `lossyear` band contains values 0-23, where 0 means no loss and 1-23 represent years 2001-2023.

In [ ]:
# Your code here for Exercise 1
# Visualize the lossyear band

# Step 1: Examine the lossyear band
print(lossyear)

# Step 2: Create a basic plot
# YOUR CODE HERE

# Step 3: Create a more advanced plot with ggplot2
# YOUR CODE HERE

## Part 2: Understanding and Visualizing Vector Data

### What is Vector Data?

Vector data represents geographic features using geometric shapes:
- **Points**: Single locations (e.g., cities, sampling locations)
- **Lines**: Connected points (e.g., roads, rivers)
- **Polygons**: Enclosed areas (e.g., administrative boundaries, lakes)

Vector data is ideal for:
- Discrete features with exact boundaries
- Data with attributes (e.g., population of cities, names of rivers)
- Network analysis (e.g., routing)

### Loading Vector Data (Shapefiles)

Let's load the province boundaries of the Philippines:

In [ ]:
library(sf)
# Create a temporary directory to store shapefile components
temp_dir <- tempdir()

# Base URL for the shapefiles
base_url <- "https://github.com/yse-eds-cert/yse-eds-cert-classroom-code-along-notebooks-code_along_notebooks/raw/refs/heads/main/course-4-environmental-analysis/geospatial-data/simplified/"

# File names for the provinces shapefile components
province_files <- c("provinces_simplified.shp", "provinces_simplified.shx", "provinces_simplified.dbf", "provinces_simplified.prj")

# Download each file
for (file_name in province_files) {
  download.file(paste0(base_url, file_name), destfile = file.path(temp_dir, file_name), mode = "wb")
}

# Load Province Shapefiles from the temporary directory
provinces <- st_read(file.path(temp_dir, "provinces_simplified.shp"), quiet = TRUE)

# Look at the first few rows
print(provinces)

### Working with Vector Data in R: The `sf` Package

The `sf` (Simple Features) package is the modern standard for working with vector data in R:

- **Simple Features**: Implements the OGC Simple Features standard
- **Integration**: Works well with the tidyverse ecosystem
- **Performance**: Efficient handling of spatial operations
- **Visualization**: Easy plotting with ggplot2 using `geom_sf()`

Key functions in `sf`:
- `st_read()`: Read vector data from files
- `st_write()`: Write vector data to files
- `st_transform()`: Change coordinate reference systems
- `st_buffer()`, `st_intersection()`, `st_union()`: Spatial operations

The `sf` package represents spatial data as simple feature collections, which are essentially data frames with a special geometry column.



Let's also load the Philippines regions shapefile--Regions are the larger administrative units that contain provinces. You write code to load the regions--they are in "../../geospatial-data/simplified/regions_simplified.shp"

In [ ]:
# Load Philippines Regions Shapefile
# YOUR CODE HERE

# Look at the first few rows
print(regions)



### Understanding Vector Data Properties

Let's examine the properties of our vector data:

In [ ]:
# Get information about the shapefile
cat("Number of features:", nrow(provinces), "\n")
cat("Geometry type:", st_geometry_type(provinces)[1], "\n")
cat("Coordinate Reference System:", st_crs(provinces)$input, "\n")

# Check column names (attributes)
cat("Available attributes:", paste(names(provinces), collapse=", "), "\n")

### Basic Vector Data Visualization

Let's create a simple map of all provinces in the Philippines:

In [ ]:
# Plot all provinces
ggplot(provinces) +
  geom_sf() +
  theme_minimal() +
  labs(title = "Provinces of the Philippines")

### Now you try plotting the regions

In [ ]:
# now you plot the regions
# YOUR CODE HERE



## Summary of Lesson 2: Loading and Visualizing Spatial Data in R

### In this lesson, we've covered the fundamental concepts and techniques for working with spatial data in R, focusing on both raster and vector data types. Here's what we've learned:

1. **Understanding Spatial Data Types**:
   - Explored the basic structure and properties of raster and vector data
   - Learned how these different data types represent spatial information

2. **Loading Spatial Data**:
   - Used the `sf` package to load and manipulate vector data (shapefiles of Philippines provinces and regions)
   - Used the `terra` package to work with raster data (deforestation data)

3. **Exploring Spatial Metadata**:
   - Examined properties of spatial datasets including coordinate reference systems
   - Accessed and inspected attributes of vector data
   - Explored resolution and extent of raster data

4. **Visualizing Spatial Data**:
   - Created maps of vector data using `ggplot2` with `geom_sf()`
   - Visualized raster data using appropriate plotting functions

These skills provide a foundation for more advanced spatial analysis and visualization in R.


## Resources

- **`sf` package (Simple Features for R)**:
  - [CRAN page](https://cran.r-project.org/web/packages/sf/index.html)
  - [GitHub repository](https://r-spatial.github.io/sf/)
  - [Vignettes and tutorials](https://r-spatial.github.io/sf/articles/sf1.html)

- **`terra` package (Spatial Data Analysis)**:
  - [CRAN page](https://cran.r-project.org/web/packages/terra/index.html)
  - [GitHub repository](https://github.com/rspatial/terra)
  - [Package documentation](https://rspatial.org/pkg/terra/)

- **`ggplot2` package (Data Visualization)**:
  - [ggplot2 website](https://ggplot2.tidyverse.org/)
  - [Documentation](https://ggplot2.tidyverse.org/reference/index.html)

- **`tidyterra` package (tidy-`terra` integration)**:
  - [CRAN page](https://cran.r-project.org/web/packages/tidyterra/index.html)
  - [GitHub repository](https://github.com/dieghernan/tidyterra)

- **General R Spatial Resources**:
  - [Geocomputation with R](https://geocompr.robinlovelace.net/)
  - [R Spatial](https://rspatial.org/)